> 📓 **Lesson 1.9 — Part 3 of 4: Aggregation & Reporting**
>
> This notebook was split out of the original single `eda_advanced.ipynb` so each part can be opened and run on its own. If you are starting here rather than at Part 1, run the **Setup** cell below first — it loads the same data used throughout Lesson 1.9.
>
> Other notebooks in this set: `Part_1_time_series.ipynb`, `Part_2_data_integration.ipynb`, `Part_4_table_to_decision.ipynb`

# Lesson 1.9: EDA Advanced — Data Wrangling & Analysis

Lesson 1.8 asked *"can I trust this data?"*. This lesson asks the next question:
**what is the pattern, and what should we do about it?**

Clean rows on their own answer nothing. You have to put time on the index, join in the tables that
give the rows meaning, reshape them, and group them. That is the whole job here.

**Structure — the four learning outcomes, in order:**
* **Part 1: Time Series** — *parse* dates, then resample and roll them.
* **Part 2: Data Integration** — *merge* tables, and convert wide ↔ long.
* **Part 3: Aggregation & Reporting** — *aggregate* with `groupby`, `pivot_table`, `crosstab`.
* **Part 4: From Table to Decision** — *apply* all of it to answer the owner's actual question.

**How to read the code cells:** read the `# 👉` comment above each line before you run the cell. The comment says
what the line does in plain English; the output shows you it happened.


> **🧭 Today's flow — 180 minutes.** One business problem, four learning outcomes, in order:
>
> | | Section | Learning outcome | Time |
> |---|---|---|---|
> | — | Setup + why this matters | | 5 min |
> | **Part 1** | Time Series | **Parse** dates; `resample`, `rolling`, `shift` | 45 min |
> | ☕ | *Break* | | 10 min |
> | **Part 2** | Data Integration | **Merge** tables; `melt` / `pivot` (wide ↔ long) | 45 min |
> | ☕ | *Break* | | 10 min |
> | **Part 3** | Aggregation & Reporting | **Aggregate**: `groupby`, `pivot_table`, `crosstab` | 45 min |
> | **Part 4** | From Table to Decision | **Apply** split-apply-combine to the real question | 20 min |
>
> **The spine:** one business problem — *The Daily Grind*, a four-outlet café chain — and one main
> file, `data/daily_sales.csv`, from start to finish. Small hand-built tables appear alongside as
> *drills*: they isolate one method so you can see exactly what it does.
>
> Each of Parts 1–3 ends with a **🛠️ Group Exercise**. Deep dives live in `reference.md`.


### The four beats of every summary

Lesson 1.8 gave you four beats for every fix: **find it → decide → apply → verify.**
Summarising has its own four, and every table we build today follows them:

| Beat | Ask yourself | |
|---|---|---|
| **1. Question** | What decision does this number serve? | *Renew the Marina Bay lease — yes or no?* |
| **2. Grain** | One row per **what**? | *One row per outlet, per month* |
| **3. Aggregation** | Sum, mean or count — and **why that one**? | *Sum for totals, mean for efficiency* |
| **4. Check** | Does the total still tie back? | *Grouped total == ungrouped total* |

Beat 4 is the one everyone skips. In Part 2 it catches a join that silently deletes $61,310.


### Setup

Import the libraries, then load the file we will use all session.


In [ ]:
# 👉 Two toolkits. `pd` and `np` are just short nicknames, so we can type `pd.something`
#    instead of `pandas.something`. Run this cell first, every session.
import pandas as pd
import numpy as np

# 👉 Housekeeping only. Pandas renames a few option strings between versions and shouts
#    about it; this keeps those notices out of our output. Nothing to learn here.
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
# 👉 The spine. One row per outlet, per day, per part of the day (Morning/Midday/Evening).
#    18 months of trading for a four-outlet café chain, plus a pop-up kiosk.
#    `parse_dates=["date"]` tells pandas: this column is not text, it is a date. More on that in 1.1.
sales = pd.read_csv("../data/daily_sales.csv", parse_dates=["date"])

# 👉 Same as Part 1: a `month` column, so later groupbys can use it as a key.
sales["month"] = sales["date"].dt.to_period("M")   # 2024-01, 2024-02, ...

sales.head()


In [ ]:
# 👉 The 1.8 habit still applies: look before you leap. Shape, types, holes.
print("rows, columns:", sales.shape)
sales.info()


In [ ]:
# 👉 Section 3.5 reuses `staffed` and `wide_view`, both built in Part 2 by joining outlets/roster
#    and reshaping targets. Rebuilt here so this notebook runs standalone -- see
#    Part_2_data_integration.ipynb sections 2.2 and 2.4 for how each one is built, step by step.
outlets = pd.read_csv("../data/outlets.csv", parse_dates=["opened_date"])
roster = pd.read_csv("../data/roster.csv", parse_dates=["week_start"])

monthly = sales.groupby(["outlet_id", "month"])["revenue_sgd"].sum().round(2).reset_index()
monthly["month_str"] = monthly["month"].astype(str)
wide_view = monthly.pivot(index="month_str", columns="outlet_id", values="revenue_sgd").round(0)

weekly_sales = (
    sales.groupby(["outlet_id", pd.Grouper(key="date", freq="W-MON", label="left")])["revenue_sgd"]
    .sum()
    .reset_index()
    .rename(columns={"date": "week_start"})
)
staffed = weekly_sales.merge(roster, on=["outlet_id", "week_start"], how="inner", validate="one_to_one")
staffed["rev_per_staff_hour"] = (staffed["revenue_sgd"] / staffed["staff_hours"]).round(2)


### 🎬 Why this matters — the flat line that hides everything

**The situation.** *The Daily Grind* runs four cafés in Singapore. Revenue has been flat for two
quarters. The Marina Bay lease is up for renewal this month, rent is $9,600, and the owner has to
sign or walk away. She sends you the sales export and asks one question: **what is going on?**

Run the next two cells. The first is the number she already has. The second is the same number,
split by outlet.

> Do not worry about how these two lines work yet — that is Part 1 and Part 3. Just read the output.


In [ ]:
# 👉 Total revenue per quarter for the whole chain -- the headline the owner already has.
#    (`.to_period("Q")` labels each date with its calendar quarter.)
chain_by_quarter = sales.groupby(sales["date"].dt.to_period("Q"))["revenue_sgd"].sum().round(0)

chain_by_quarter


In [ ]:
# 👉 The same revenue, but one column per outlet. Same data. Same period. Different question.
by_outlet = sales.pivot_table(
    index=sales["date"].dt.to_period("Q"),   # down the side: quarter
    columns="outlet_id",                     # across the top: outlet
    values="revenue_sgd",                    # the number in the middle
    aggfunc="sum",                           # how to squash it: add it up
).round(0)

by_outlet


**Read the second table.** OUT-03 falls from about \$138k a quarter to about \$100k. OUT-04 climbs
from about \$96k to about \$131k. One is dying, one is growing, and they move by almost the same
amount — so the chain total barely twitches.

The flat line was never the story. It was **two opposite stories cancelling out**.

No amount of cleaning would have found this. Cleaning gives you rows you can trust; only grouping
turns them into an answer. Three more things you cannot see yet, and will by the end of the session:

1. OUT-03's fall is not a slope. It is a **step**, on one specific week. (Part 1)
2. There is a fifth outlet in this file that does not exist in the outlet list. (Part 2)
3. OUT-03 is still staffed for the revenue it used to make. (Part 3)

Write those three down. We will tick them off.


## Part 3: Aggregation & Reporting

**Learning outcome 3:** *Aggregate data using `groupby`, pivot tables and cross-tabulations to
generate summary reports.*

**Goal:** turn 6,840 rows into the four or five numbers that a decision actually needs.

⏱️ ~45 min including Group Exercise 3


### 3.1: `groupby` — split, apply, combine

Three steps, always in this order:

1. **Split** the rows into groups by some key.
2. **Apply** a calculation to each group, independently.
3. **Combine** the answers into one table.


In [ ]:
# 👉 A drill you can check by hand: six rows, two groups.
drill = pd.DataFrame({
    "outlet": ["A", "A", "A", "B", "B", "B"],
    "daypart": ["Morning", "Midday", "Evening"] * 2,
    "revenue": [100, 60, 40, 30, 70, 90],
})

drill


In [ ]:
# 👉 Split by outlet, apply sum, combine. A: 100+60+40 = 200. B: 30+70+90 = 190.
drill.groupby("outlet")["revenue"].sum()


In [ ]:
# 👉 Group by TWO keys and you get one row per combination -- a MultiIndex (hierarchical index).
drill.groupby(["outlet", "daypart"])["revenue"].sum()


In [ ]:
# 👉 `.unstack()` lifts the last index level up into columns. This is exactly a pivot table,
#    built from a groupby. Same numbers, more readable shape.
drill.groupby(["outlet", "daypart"])["revenue"].sum().unstack()


In [ ]:
# 👉 Now the real thing. Total revenue per outlet across 18 months.
sales.groupby("outlet_id")["revenue_sgd"].sum().round(0)


### 3.2: `.agg()` — several questions in one pass

`.agg()` applies more than one function at a time, and lets you name the output columns. One pass
over the data, one readable table.


In [ ]:
# 👉 Named aggregation: each argument is `new_column=("source_column", "function")`.
#    Read it as a specification of the report you want.
outlet_report = sales.groupby("outlet_id").agg(
    revenue=("revenue_sgd", "sum"),
    tickets=("tickets", "sum"),
    trading_days=("date", "nunique"),
    best_day=("revenue_sgd", "max"),
)

# 👉 Derived columns come after the aggregation, from the aggregated numbers.
outlet_report["avg_ticket"] = (outlet_report["revenue"] / outlet_report["tickets"]).round(2)
outlet_report["revenue_per_day"] = (outlet_report["revenue"] / outlet_report["trading_days"]).round(0)
outlet_report["revenue"] = outlet_report["revenue"].round(0)
outlet_report["best_day"] = outlet_report["best_day"].round(0)

outlet_report


> **`nunique` vs `count` vs `size`.** `count` counts non-null values, `size` counts rows including
> nulls, and `nunique` counts distinct values. `trading_days` above had to be `nunique` — there are
> three rows per day, so `count` would have said 1,641 trading days in an 18-month period.


### 3.3: `pivot_table` — the two-dimensional summary

A pivot table is a `groupby` on two keys with the second one spread across the top. Four decisions:
**index** (down the side), **columns** (across the top), **values** (the number), **aggfunc** (how
to squash it).


In [ ]:
# 👉 Outlet down the side, daypart across the top, revenue in the middle, added up.
#    `margins=True` adds the "All" row and column -- the totals, and a free beat-4 check.
daypart_mix = sales.pivot_table(
    index="outlet_id",
    columns="daypart",
    values="revenue_sgd",
    aggfunc="sum",
    margins=True,
).round(0)

daypart_mix


In [ ]:
# 👉 Absolute dollars hide the pattern because the outlets are different sizes. Convert each row
#    to percentages of its own total: `div` divides, `axis=0` means "row by row".
mix_pct = (
    daypart_mix.drop(index="All").drop(columns="All")
    .div(daypart_mix.drop(index="All")["All"], axis=0)
    .mul(100).round(1)
)

mix_pct[["Morning", "Midday", "Evening"]]


> **Read the Morning column.** OUT-01 and OUT-03 make about half their money before 11am — office
> workers on the way in. OUT-04 makes barely a third in the morning and a third in the evening —
> that is a neighbourhood, not a commute. Same chain, two different businesses, and the marketing
> that works on one will not work on the other.


In [ ]:
# 👉 pivot_table takes several aggfuncs at once. Note what happens to the column headers:
#    they become two levels deep (aggfunc, then daypart).
sales.pivot_table(
    index="outlet_id", columns="daypart", values="revenue_sgd", aggfunc=["mean", "count"]
).round(0)


### 3.4: `crosstab` — counting combinations

`crosstab` is a pivot table specialised for **frequency**: how often does each combination occur?
This needs one row per event, so we switch to the ticket-level file for one week.


In [ ]:
# 👉 One row per till receipt, for the week of 16-22 June 2025.
tickets = pd.read_csv("../data/tickets_week.csv", parse_dates=["txn_datetime"])

tickets.head(3)


In [ ]:
# 👉 How many receipts for each payment method, in each part of the day? Plain counts.
pd.crosstab(tickets["payment_method"], tickets["daypart"])


In [ ]:
# 👉 Counts are hard to compare between columns of different sizes. `normalize="columns"`
#    turns each column into proportions of itself, so the columns are comparable.
(pd.crosstab(tickets["payment_method"], tickets["daypart"], normalize="columns") * 100).round(1)


In [ ]:
# 👉 crosstab can aggregate a value instead of counting, with `values=` + `aggfunc=`.
#    Average ticket size by outlet and category -- who is buying the expensive things?
pd.crosstab(
    tickets["outlet_id"], tickets["category"], values=tickets["amount_sgd"], aggfunc="mean"
).round(2)


### 3.5: Correlation — do two columns move together?

`.corr()` gives a number between -1 and 1 and, unlike covariance, it does not depend on the units.


In [ ]:
# 👉 Does staffing track revenue? Use the weekly table we merged in 2.2.
print("correlation, revenue vs staff hours:", staffed["revenue_sgd"].corr(staffed["staff_hours"]).round(3))

# 👉 Covariance answers the same "do they move together" question, but its size depends on the
#    units, so on its own it is uninterpretable. Correlation is covariance, normalised.
print("covariance (same relationship, unreadable scale):", round(staffed["revenue_sgd"].cov(staffed["staff_hours"]), 1))


In [ ]:
# 👉 On a DataFrame, `.corr()` gives every pair at once. Here: do the outlets' monthly revenues
#    move together? Compare the OUT-03 / OUT-04 cell with the others.
wide_view.corr().round(2)


> **Careful.** OUT-03 and OUT-04 are strongly *negatively* correlated, and nothing OUT-04 does
> takes money from OUT-03 — they are 8km apart with different customers. One is declining and the
> other is growing over the same 18 months, and correlation cannot tell that apart from a cause.
> Use it to *find candidates to investigate*, never as the finding itself.
>
> And look at the OUT-05 row: the kiosk traded for **three months**, so every number in it comes
> from three data points. A correlation computed from three points is a coincidence with a decimal
> place on it. Always ask how many observations are behind a correlation before you quote it.


In [ ]:
# 👉 The staffing question, properly. Revenue per staff hour, per outlet, for the last two quarters.
recent = staffed[staffed["week_start"] >= "2025-01-01"]

staffing = recent.groupby("outlet_id").agg(
    revenue=("revenue_sgd", "sum"),
    staff_hours=("staff_hours", "sum"),
)
staffing["rev_per_staff_hour"] = (staffing["revenue"] / staffing["staff_hours"]).round(2)
staffing["revenue"] = staffing["revenue"].round(0)
staffing["staff_hours"] = staffing["staff_hours"].round(0)

staffing.sort_values("rev_per_staff_hour")


> **Tick off finding #3.** Marina Bay earns the fewest dollars per staff hour of the four real
> outlets -- about \$20 an hour against \$25--27. Its revenue fell in November; its roster did not. The rota is still sized for the café
> it used to be — and that is a fixable problem worth real money, quite separate from the lease.


### 🛠️ Group Exercise 3 — Aggregation (8 min)

Using `tickets`, build a crosstab of `category` against `daypart`, normalised by column. Which category's share is biggest in the Evening?

---

## ✅ Sample Solution

Try the exercise yourself first — this is *a* solution, not *the* solution. If your code reaches the same answer a different way, it is right.

**Category share by daypart.**

In [ ]:
(pd.crosstab(tickets["category"], tickets["daypart"], normalize="columns") * 100).round(1)

Coffee is the biggest share in every daypart, but the Evening column is the *least* coffee-heavy: ~37.5% against ~55.8% in the Morning. Food is second in the Evening (~26.9%) and nearly absent in the Morning (~4.0%). `normalize="columns"` makes each column sum to 100, so you are comparing **shares within a daypart** — not how busy the dayparts are. A share can rise while the count falls.

---

**⏭ Up next — Part 4: From Table to Decision.**

📂 Open `Part_4_table_to_decision.ipynb` to continue.